# Inspeccionar pesos y barreras progresivas

**Pregunta:** ¿Cómo afectan los pesos y la detención con barrera progresiva a la configuración de magnetita seleccionada?

Nivel: 2. Requisitos previos: lecciones anteriores del tema. Lectura: 30 minutos.



## Objetivos

El circuito de magnetita fina aporta un problema unidimensional de molienda. Se registran estado y costo exacto para pesos 0, 0.5 y 1 con límites idénticos. Factibilidad, convergencia y acuerdo físico son resultados distintos y no se presuponen.

## Preparación

```bash
pip install numpy==2.2.6 scipy==1.17.1 pandas==2.2.3 matplotlib==3.11.2 nbclient==0.11.0 nbformat==5.11.1 ipykernel==7.4.0 SALib==1.6.0 scikit-learn==1.5.2 ruff==0.15.18 pytest==9.1.1
```

El pipeline OreFlow se importa desde `data-pipeline/`; este notebook nunca instala paquetes.

Importar las herramientas fijadas y resolver las entradas relativas a la carpeta del notebook.

In [ ]:
# Importar herramientas de análisis y registrar sus versiones exactas.
import importlib.metadata
import json
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

repo = Path("../../../").resolve()
sys.path.insert(0, str(repo / "data-pipeline"))
for package in ("numpy", "scipy", "pandas", "matplotlib", "nbclient", "nbformat", "ipykernel"):
    print(package, importlib.metadata.version(package))
legend_style = dict(loc="lower center", bbox_to_anchor=(0.5, 1.02), frameon=False)


## Datos

Responsable: autores de OreFlow, licencia MIT. Las entradas son el caso `iron_magnetite_fine` del catálogo y `data/derived/contract/operating_contract.json`. No se usa un conjunto medido de planta ni modelo económico. Fuente: https://github.com/fsantibanezleal/CAOS_OreFlow.

## Método

Tres búsquedas completas mantienen circuito, contrato y comienzos mientras cambian peso de metal a 0, 0.5 y 1. El peso energético es complementario. Son preferencias físicas normalizadas, no precios. Cada corrida conserva estado, holguras, convergencia y detención por comienzo. Los extremos físicos infactibles se identifican en la figura; los rechazados permanecen en tabla sin coordenadas inventadas. El punto intermedio recibe auditoría independiente, separando preferencia, factibilidad y regla de malla.

Prepara magnetita para examinar construcción del objetivo, no costo del filtro. Masa recuperada y energía nominales definen constantes separadas de normalización. Sin ellas, una suma ponderada mezclaría arbitrariamente toneladas por hora y kWh por tonelada. Los experimentos comparten decisión acotada y restricciones. Así un cambio de configuración se atribuye a preferencias, no a redefinir el proceso.

In [ ]:
# Usar el circuito de magnetita de autoría propia como búsqueda acotada de una dimensión.
from pipeline.cases.catalog import CASE_BY_ID
from pipeline.engine.circuit import simulate
from pipeline.methods.optimization import optimize, Problem
from pipeline.methods.pattern_search import pattern_search
from pipeline.engine.constants import constant

case = CASE_BY_ID["iron_magnetite_fine"]
contract = json.loads(
    (repo / "data/derived/contract/operating_contract.json").read_text(encoding="utf-8")
)
base = simulate(case.ore, case.plant, case.nominal)
print("Recuperación base (%):", round(base.metrics["recovery_pct"], 3))
print("Límites de decisión:", contract["cases"][case.id]["inputs"]["target_p80_um"])


Resolver primero pesos iguales. Inspeccionar detención por comienzo y holguras de ley, potencia y agua. La barrera permite avanzar hacia factibilidad desde un comienzo infactible, en lugar de asignar a todos esos puntos una penalización indistinguible. Ordenar objetivo y violación sigue exigiendo motor exacto. Un extremo útil necesita estado admisible y condición de detención registrada.

In [ ]:
# Ejecutar la búsqueda multicomienzo de producción sin filtro aprendido, con pesos iguales en metal
# recuperado y energía normalizados.
plain = optimize(case, case.nominal, contract, weight=0.5, path=False, screen=None)
print("Estado sin filtro:", plain["status"])
print("Evaluaciones sin filtro:", plain["evaluations"])
print("Motivos de detención:", [r["stop"] for r in plain["starts"]])
plain_end = plain.get("optimum") or plain.get("least_violating") or plain["base"]
display(pd.Series(plain_end["slacks"], name="Holgura de restricción").to_frame())


Ejecutar preferencia de solo metal. El peso no elimina restricciones ni concede potencia adicional; retira energía del objetivo. El extremo solo aumenta masa dentro de la misma región física. Conservar consumo energético en la comparación: mejor para el solver significa mejorar el objetivo pedido, no representar todas las prioridades operativas.

In [ ]:
# Cambiar el objetivo a solo metal recuperado conservando límites, comienzos y restricciones
# físicas.
metal_search = optimize(case, case.nominal, contract, weight=1.0, path=False, screen=None)
metal_end = (
    metal_search.get("optimum") or metal_search.get("least_violating") or metal_search["base"]
)
print("Preferencia:", 1.0)
print("Metal recuperado (t/h):", round(metal_end["recovered_tph"], 6))
print("Energía (kWh/t):", round(metal_end["values"]["energy_kwh_t"], 3))


Ejecutar preferencia de solo energía con iguales comienzos. Minimizar energía conserva especificación del concentrado y límites físicos; no autoriza dejar de moler sin importar calidad. Comparar toneladas recuperadas para mostrar el sacrificio. Tres pesos son una muestra pequeña de preferencias, no una frontera Pareto exhaustiva ni utilidad económica ajustada.

In [ ]:
# Cambiar el objetivo a solo energía específica conservando límites, comienzos y restricciones
# físicas.
energy_search = optimize(case, case.nominal, contract, weight=0.0, path=False, screen=None)
energy_end = (
    energy_search.get("optimum") or energy_search.get("least_violating") or energy_search["base"]
)
print("Preferencia:", 0.0)
print("Metal recuperado (t/h):", round(energy_end["recovered_tph"], 6))
print("Energía (kWh/t):", round(energy_end["values"]["energy_kwh_t"], 3))


Auditar la preferencia intermedia antes de compararla con extremos de metal y energía. Solo un estado definido aporta recuperación y energía; resolver circuito y cumplir restricciones siguen siendo preguntas distintas.

In [ ]:
# Audit the reported unscreened endpoint only when the engine can define that state.
from pipeline.engine.model import InfeasibleState

plain_point = case.nominal.with_values(**plain_end["decisions"])
plain_check = None
if plain["status"] != "base_refused":
    try:
        plain_check = simulate(case.ore, case.plant, plain_point)
    except InfeasibleState as refusal:
        print("Rechazo de auditoría sin filtro:", refusal.code)
print("Auditoría sin filtro disponible:", plain_check is not None)
if plain_check is not None:
    print("Recuperación exacta sin filtro (%):", round(plain_check.metrics["recovery_pct"], 3))
    print(
        "Energía exacta sin filtro (kWh/t):",
        round(plain_check.metrics["specific_energy_total_kwh_t"], 3),
    )


Inspeccionar historias de objetivo y malla del comienzo de pesos iguales. Las mejoras pueden mantener o expandir malla y los fracasos contraerla. Los paneles distinguen evaluaciones e iteraciones. Como los comienzos pueden reutilizar caché del problema, sumar sus historias no reemplaza el conteo total de llamadas únicas. Interpretar cada conteo según la cantidad registrada.

In [ ]:
# Graficar el objetivo incumbente y la malla del comienzo seleccionado sin filtro.
plt.rcParams.update({"font.size": 22, "axes.titlepad": 8})
history = pd.DataFrame(
    plain.get("trace", []), columns=["Outcome", "Mesh", "Barrier", "Objective", "Evaluations"]
)
fig, axes = plt.subplots(2, 1, figsize=(6.4, 4.8))
axes[0].plot(history["Evaluations"], history["Objective"], marker=".")
axes[0].set(xlabel="Llamadas exactas (n)", ylabel="Objetivo (1)", title="Progreso del objetivo")
axes[1].step(range(len(history)), history["Mesh"], where="post")
axes[1].set(xlabel="Iteración (n)", ylabel="Malla (1)", title="Malla de búsqueda")
fig.subplots_adjust(left=0.24, right=0.96, bottom=0.15, top=0.91, hspace=1.0)


La traza sostiene una afirmación acotada: el comienzo seleccionado terminó bajo la regla impresa después de su secuencia registrada. No demuestra óptimo global ni robustez a otro mineral o índice incierto. Un objetivo plano final puede reflejar tolerancia de mejora, no una meseta física exacta. Conservar motivo de detención al citar recuperación y energía terminales.

Graficar los tres extremos en coordenadas energéticas y de metal. Una puntuación ponderada oculta la relación; unidades físicas hacen visible el compromiso. Conectar marcas solo guía lectura, sin calcular pesos intermedios, garantizar convexidad ni establecer factibilidad sobre toda la línea. Las holguras siguen siendo evidencia de admisibilidad de cada candidato calculado.

In [ ]:
# Preserve each run status in the physical preference plot.
plt.rcParams.update({"font.size": 18, "axes.titlepad": 80})
preference_runs = (("Igual", plain), ("Metal", metal_search), ("Energía", energy_search))
fig, ax = plt.subplots(figsize=(6.4, 3.8))
for label, run in preference_runs:
    end = run.get("optimum") or run.get("least_violating") or run["base"]
    if run["status"] != "base_refused" and end["values"]:
        marker = "o" if end["feasible"] else "x"
        ax.scatter(
            end["values"]["energy_kwh_t"],
            end["recovered_tph"],
            marker=marker,
            label=label + " [" + run["status"] + "]",
        )
ax.set(
    xlabel="Energía específica\n(kWh/t)",
    ylabel="Metal recuperado\n(t/h)",
    title="Extremos de preferencias calculados",
)
ax.legend(ncol=2, **legend_style)
fig.subplots_adjust(left=0.22, right=0.96, bottom=0.23, top=0.61)


Cada marca representa un extremo calculado y su leyenda conserva el estado. Las cruces identifican resultados resolubles pero infactibles; los rechazos no aportan puntos inventados. La tabla posterior conserva todas las corridas y salidas ausentes. No hay una línea que sugiera frontera factible interpolada ni precios económicos para electricidad o metal.

La violación se forma con holguras normalizadas negativas fuera de la tolerancia de factibilidad. Sus cuadrados sumados constituyen la cantidad de barrera que usa la búsqueda. Es distinta del objetivo energía/metal: un candidato puede mejorar factibilidad y empeorar preferencia. Así se explica gastar evaluaciones sin elevar el incumbente factible. La caché multicomienzo registra vectores físicos únicos compartidos entre comienzos, mientras la traza seleccionada cuenta trabajo dentro de uno. No inferir el total multiplicando longitud por cantidad de comienzos. Si cambias límites contractuales, también cambias la conversión entre coordenadas unitarias y molienda física; ese cambio exigiría otro experimento y no sería comparable directamente con los extremos actuales.

El peso cero minimiza energía entre candidatos admisibles, no maximiza una supuesta eficiencia económica. El peso uno maximiza metal normalizado y no convierte tonelaje en ingreso. El punto intermedio permite observar un compromiso físico calculado. Para convertirlo en preferencia de negocio faltan precios, costos variables, límites ambientales y una calibración de sus incertidumbres; esos términos no se inventan al interpretar la figura.

Conservar las tres preferencias en tabla común con sus propias holguras y detenciones. Una coordenada atractiva no oculta infactibilidad; el rechazo mantiene estado y decisión de entrada sin inventar salidas físicas.

In [ ]:
# Informar cada corrida antes de comparar costo o preferencias.
runs = {"Pesos iguales": plain, "Solo metal": metal_search, "Solo energía": energy_search}
endpoint_rows = []
for label, run in runs.items():
    end = run.get("optimum") or run.get("least_violating") or run["base"]
    physical = run["status"] != "base_refused" and bool(end["values"])
    endpoint_rows.append(
        {
            "Corrida": label,
            "Estado": run["status"],
            "Factible": end["feasible"],
            "Convergencia": run.get("converged"),
            "Detenciones": ", ".join((r["stop"] for r in run["starts"])),
            "Decisiones": end["decisions"],
            "Holguras": end["slacks"],
            "Llamadas exactas": run["evaluations"],
            "Metal recuperado (t/h)": end["recovered_tph"] if physical else None,
            "Recuperación (%)": end["recovery_pct"] if physical else None,
            "Energía específica (kWh/t)": end["values"].get("energy_kwh_t") if physical else None,
        }
    )
endpoints = pd.DataFrame(endpoint_rows)
with pd.option_context("display.max_colwidth", None, "display.max_columns", None):
    display(endpoints)


## Resultados

**Respuesta:** Con pesos iguales el objetivo es <!-- result:P80 objetivo informado (µm) -->0<!-- /result --> µm y factibilidad <!-- result:Extremo factible (1=sí) -->0<!-- /result -->. Los indicadores de metal y energía son <!-- result:Extremo de metal factible (1=sí) -->0<!-- /result --> y <!-- result:Extremo de energía factible (1=sí) -->0<!-- /result --> (1=sí). Salidas físicas y holguras rotuladas muestran el compromiso; detención por comienzo y convergencia califican término. Salidas ausentes no se convierten en ceros graficados.

Resumir objetivo intermedio e indicadores de factibilidad de las tres preferencias después de inspeccionar estados.

In [ ]:
# Resumir objetivo intermedio e indicadores de factibilidad de las tres preferencias después de
# inspeccionar estados.
answer_values = {
    "Conteo sin filtro": plain["evaluations"],
    "P80 objetivo informado (µm)": plain_end["decisions"]["target_p80_um"],
    "Extremo factible (1=sí)": int(plain_end["feasible"]),
    "Convergencia informada (1=sí)": int(plain.get("converged") is True),
}
answer_values.update(
    {
        "Extremo de metal factible (1=sí)": int(metal_end["feasible"]),
        "Extremo de energía factible (1=sí)": int(energy_end["feasible"]),
    }
)
answer_values = {label: round(float(number), 8) for label, number in answer_values.items()}
print(json.dumps(answer_values, ensure_ascii=False))


## Resumen

Los tres extremos conservan factibilidad y detención individuales. Diferencias de masa y energía proceden de pesos normalizados, sin afirmar utilidad económica ni frontera factible interpolada.

## Referencias

- https://doi.org/10.1137/S1052623493250780
- https://doi.org/10.1137/070692662